In [ ]:
import importlib
import subprocess
import sys

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"📦 Cài đặt thư viện: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

ensure_package("torch")
ensure_package("transformers")
ensure_package("tqdm")
ensure_package("pandas")
ensure_package("numpy")
ensure_package("scikit-learn", "sklearn")

import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    from vn_preprocessor import VNPreprocessor
except ImportError:
    print("❗Lỗi: Không tìm thấy `vn_preprocessor.py`. Vui lòng upload file này vào thư mục làm việc.")
    raise

try:
    from LDA_classifier import LDAClassifier
except ImportError:
    print("❗Lỗi: Không tìm thấy `LDA_classifier.py`. Vui lòng upload file này vào thư mục làm việc.")
    raise

try:
    from TFIDF_vectorlizer import TFIDFVectorizer
except ImportError:
    print("❗Lỗi: Không tìm thấy `TFIDF_vectorlizer.py`. Vui lòng upload file này vào thư mục làm việc.")
    raise


Load dataset

In [ ]:
ds = load_dataset(
    "uitnlp/vietnamese_students_feedback", revision="refs/convert/parquet"
)

Chia dataset thành 3 tập train, validation và test

In [ ]:
df_train = pd.DataFrame(ds["train"][:])
df_val = pd.DataFrame(ds["validation"][:])
df_test = pd.DataFrame(ds["test"][:])

prep = VNPreprocessor(
    text_col="sentence",
    analyzer="word",
    use_underthesea=True,
    add_clean=True,
    add_no_stop=True,
    add_tokens=False,
    add_lengths=True,
)

prep_raw = VNPreprocessor(
    text_col="sentence",
    analyzer="word",
    use_underthesea=False,
    add_clean=True,
    add_no_stop=True,
    add_tokens=False,
    add_lengths=True,
)

Vectorize dataset

In [ ]:
vectorlizer = TFIDFVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
)

## vectorlizer cho tập train
df_underthesea_train = vectorlizer.fit_transform(
    df_underthesea_train, text_column="sentence_nostop", new_column="vector"
)
df_raw_train = vectorlizer.transform(
    df_raw_train, text_column="sentence_nostop", new_column="vector"
)

# Ghép thành DataFrame
df_display = pd.DataFrame(
    {
        "Raw sentence": df_raw_train["sentence_nostop"],
        "Underthesea sentence": df_underthesea_train["sentence_nostop"],
    }
)

df_vectorized_display = pd.DataFrame(
    {
        "Raw vector": df_raw_train["vector"],
        "Underthesea vector": df_underthesea_train["vector"],
    }
)

print("DataFrame hiển thị:")
print(df_display.head())
print("DataFrame vectorized hiển thị:")
print(df_vectorized_display.head())

## vectorlizer cho tập test
df_raw_test = vectorlizer.transform(
    df_raw_test, text_column="sentence_nostop", new_column="vector"
)
df_underthesea_test = vectorlizer.transform(
    df_underthesea_test, text_column="sentence_nostop", new_column="vector"
)

LDA Classifier

In [ ]:
### LDA Classifier
raw_lda_classifier = LDAClassifier(solver="svd")
underthesea_lda_classifier = LDAClassifier(solver="svd")


Train model

In [ ]:
# # Train model
raw_lda_classifier.fit(
    np.array(df_raw_train["vector"].tolist()),
    np.array(df_raw_train["sentiment"].tolist()),
)
underthesea_lda_classifier.fit(
    np.array(df_underthesea_train["vector"].tolist()),
    np.array(df_underthesea_train["sentiment"].tolist()),
)
print("Model đã được train thành công.")

Dự đoán và đánh giá mô hình

In [ ]:
### Dự đoán và đánh giá mô hình
## RAW
raw_actual = df_raw_test["sentiment"]
raw_accuracy = raw_lda_classifier.score(
    np.array(df_raw_test["vector"].tolist()), raw_actual
)

## UNDERTHESEA
underthesea_actual = df_underthesea_test["sentiment"]
underthesea_accuracy = underthesea_lda_classifier.score(
    np.array(df_underthesea_test["vector"].tolist()), underthesea_actual
)

So sánh độ chính xác của mô hình không sử dụng Underthesea và có sử dụng Underthesea

In [ ]:
print(f"Độ chính xác của mô hình trên tập test không underthesea: {raw_accuracy:.2f}")
print(f"Độ chính xác của mô hình trên tập test có underthesea: {underthesea_accuracy :.2f}")

Classification Report

In [ ]:
sentiment_labels = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
# 4. Classification Report
print("\nClassification Report (Raw Data):")
raw_pred = raw_lda_classifier.predict(np.array(df_raw_test["vector"].tolist()))
print(classification_report(raw_actual, raw_pred, target_names=sentiment_labels))

print("\nClassification Report (Underthesea Data):")
underthesea_pred = underthesea_lda_classifier.predict(np.array(df_underthesea_test["vector"].tolist()))
print(classification_report(underthesea_actual, underthesea_pred, target_names=sentiment_labels))


Confusion Matrix

In [ ]:
# 5. Confusion Matrix
print("\nConfusion Matrix (Raw Data):")
cm_raw = confusion_matrix(raw_actual, raw_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=sentiment_labels, yticklabels=sentiment_labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Raw Data)')
plt.show()

print("\nConfusion Matrix (Underthesea Data):")
cm_underthesea = confusion_matrix(underthesea_actual, underthesea_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_underthesea, annot=True, fmt='d', cmap='Blues', xticklabels=sentiment_labels, yticklabels=sentiment_labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Underthesea Data)')
plt.show()

Với PhoBERT

In [ ]:
from phoBERT_classifier import PhoBERTSentimentClassifier

In [ ]:
clf = PhoBERTSentimentClassifier(batch_size=16, lr=2e-5, max_len=128)
clf.prepare_dataloader(df_train, df_val, df_test)
clf.train(epochs=3)

metrics = clf.evaluate_on_test(return_report=True)
print("Accuracy:", metrics["accuracy"])
print(metrics["text_report"])       # Báo cáo dạng sklearn
print(metrics["confusion_matrix"])  # Ma trận nhầm lẫn (numpy array)

y_true = metrics["y_true"]
y_pred_phobert = metrics["y_pred"]

Lưu lại model

In [ ]:
clf.save_model("./phobert_sentiment")

Load lại model PhoBERT

In [ ]:
clf = PhoBERTSentimentClassifier()

clf.load_model("./phobert_sentiment")

Dùng model để dự đoán

In [ ]:
text = input("Nhập câu cần phân tích cảm xúc: ")
label = clf.predict(text)
print("Dự đoán:", label_map[label])
